In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

In [7]:
df=pd.read_csv("../Data/df_cleaned.csv")

#Modeling

In [8]:
train_data = df[df["date"] < "2025-10-01"].copy()

test_data = df[df["date"] >= "2025-10-01"].copy()

In [9]:
train_data=train_data.drop(columns=['date'])
test_data=test_data.drop(columns=['date'])

In [10]:
X_train = train_data.drop(columns=["posted_rate"])
y_train = train_data["posted_rate"]

X_test = test_data.drop(columns=["posted_rate"])
y_test = test_data["posted_rate"]

In [11]:
print(X_train.dtypes)
print(X_train.columns)

pickup              str
delivery            str
pickup_lat      float64
pickup_lon      float64
delivery_lat    float64
delivery_lon    float64
distance        float64
equipment           str
market_index    float64
quote_signal    float64
weight_clean    float64
month             int64
day               int64
week              int64
dtype: object
Index(['pickup', 'delivery', 'pickup_lat', 'pickup_lon', 'delivery_lat',
       'delivery_lon', 'distance', 'equipment', 'market_index', 'quote_signal',
       'weight_clean', 'month', 'day', 'week'],
      dtype='str')


#Ridge Regression

In [12]:
categorical_features = ["pickup", "delivery", "equipment"]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

ridge_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", Ridge(alpha=10))
])

ridge_model.fit(X_train, y_train)

ridge_pred = ridge_model.predict(X_test)

#XGBoost

In [14]:
xgb_model = Pipeline([
    ("preprocessor", preprocessor),
    (
        "model",
        XGBRegressor(
            n_estimators=1000,
            learning_rate=0.05,
            max_depth=8,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            random_state=42,
            n_jobs=-1
        )
    )
])

xgb_model.fit(X_train, y_train)

xgb_pred = xgb_model.predict(X_test)

#LightGBM

In [15]:
lgbm_model = Pipeline([
    ("preprocessor", preprocessor),
    (
        "model",
        LGBMRegressor(
            n_estimators=1000,
            learning_rate=0.05,
            max_depth=-1,
            num_leaves=31,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            n_jobs=-1
        )
    )
])

lgbm_model.fit(X_train, y_train)

lgbm_pred = lgbm_model.predict(X_test)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002486 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1623
[LightGBM] [Info] Number of data points in the train set: 43147, number of used features: 142
[LightGBM] [Info] Start training from score 2373.410351


#CatBoost

In [16]:
cat_features = ["pickup", "delivery", "equipment"]

In [17]:
final_model = CatBoostRegressor(
    iterations=3000,
    learning_rate=0.01,
    depth=4,
    l2_leaf_reg=5,
    random_strength=1,
    bagging_temperature=0,
    loss_function="MAE",
    eval_metric="MAE",
    random_seed=42,
    verbose=100
)

final_model.fit(
    X_train,
    y_train,
    cat_features=cat_features,
    eval_set=(X_test, y_test),
    early_stopping_rounds=150
)

0:	learn: 1117.7056820	test: 1136.9606787	best: 1136.9606787 (0)	total: 182ms	remaining: 9m 7s
100:	learn: 496.0968730	test: 511.0698807	best: 511.0698807 (100)	total: 6.17s	remaining: 2m 57s
200:	learn: 276.0459863	test: 292.0416779	best: 292.0416779 (200)	total: 11.8s	remaining: 2m 43s
300:	learn: 181.0837752	test: 195.7556157	best: 195.7556157 (300)	total: 16s	remaining: 2m 23s
400:	learn: 138.9058386	test: 153.0706537	best: 153.0706537 (400)	total: 19.6s	remaining: 2m 6s
500:	learn: 122.2150978	test: 136.3861426	best: 136.3861426 (500)	total: 23.2s	remaining: 1m 55s
600:	learn: 112.0712264	test: 126.8997691	best: 126.8997691 (600)	total: 26.8s	remaining: 1m 46s
700:	learn: 105.9808740	test: 120.4674815	best: 120.4674815 (700)	total: 30.3s	remaining: 1m 39s
800:	learn: 101.9591718	test: 116.2612683	best: 116.2612683 (800)	total: 34.6s	remaining: 1m 35s
900:	learn: 99.3004023	test: 112.7337171	best: 112.7337171 (900)	total: 39.5s	remaining: 1m 31s
1000:	learn: 97.1045935	test: 110.11

CatBoostRegressor(bagging_temperature=0, depth=4, eval_metric='MAE', iterations=3000, l2_leaf_reg=5, learning_rate=0.01, loss_function='MAE', random_seed=42, random_strength=1, verbose=100)

In [18]:
cat_pred = final_model.predict(X_test)

In [19]:
results = []


models_predictions = {
    "Ridge": ridge_pred,
    "XGBoost": xgb_pred,
    "LightGBM": lgbm_pred,
    "CatBoost": cat_pred
}

for model_name, predictions in models_predictions.items():

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    results.append({
        "Model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

results_df = pd.DataFrame(results)

results_df = results_df.sort_values("MAE")

results_df

,Model,MAE,RMSE,R2
3,CatBoost,103.854700,646.652322,0.821031
0,Ridge,166.619911,655.270010,0.816229
2,LightGBM,246.383334,694.766843,0.793407
1,XGBoost,292.568300,743.170205,0.763618
